# E7 Notebook 26C — CIC-IDS2017 Full G0 audit (safe Git checkout repair)

**Use Notebook 26C instead of 26B.** Notebook 26B had an unsafe shallow-history `fetch --depth 1` followed by `merge --ff-only`, which stopped with unrelated histories. Notebook 26C uses a **new isolated Git folder** under `/kaggle/working/rare-attack-e7-cicids-26c-source`, and **never fetches, pulls, merges, resets, force-checkouts, or deletes** a previous checkout. Existing M1/UNSW data, saved research models, and Kaggle results remain untouched.

Read-only E7 full CSV source/schema/timestamp audit only; no model fitting and no official UNSW test access. Source root `/kaggle/input/datasets/dsshayan/dscicids/` contains eight CSVs inside `TrafficLabelling ` (trailing space). Registered Monday `DD/MM/YYYY HH:MM:SS` and other-day `D/M/YYYY H:MM` formats come from user-provided source preview. Clock semantics on `Afternoon` files remain unresolved and must be flagged, not silently changed.

Each completed **aggregate-only** original G0 report is published to the dedicated E7 GitHub folder and independently verified before the next scan. Kaggle Internet and the existing `pushRare` secret are required. If you have E7 local G0 reports already, the later cells will validate/reuse them without overwriting.


## 1. Safe, independent source snapshot + 16 synthetic tests

This stage **never updates an existing checkout**. A brand-new isolated directory is cloned once and thereafter reused only if clean. All previously created Kaggle folders and research artifacts are preserved.


In [ ]:
from pathlib import Path
import subprocess
import sys

# New isolated location; the original shallow checkout is NOT touched.
REPO = Path('/kaggle/working/rare-attack-e7-cicids-26c-source')
URL = 'https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git'
if REPO.exists():
    assert (REPO/'.git').is_dir(), 'Existing 26C path is not Git; no files altered.'
    dirty = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    assert not dirty, 'Existing 26C checkout changed locally; stop without modifying it.'
    print('Reusing immutable E7-26C checkout: no fetch, pull, merge or reset.')
else:
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch','main',
                    URL,str(REPO)],check=True)

STUDY = REPO/'research/E7_CICIDS2017'
SCRIPT = STUDY/'scripts/e7_g0_external_source_audit.py'
PUBLISHER = STUDY/'scripts/e7_g0_publish.py'
REGISTRY = STUDY/'configs/e7_g0_timestamp_registry_001.json'
assert all(p.is_file() for p in (SCRIPT,PUBLISHER,REGISTRY)), 'Missing E7 files: halt.'

SOURCE_COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],
                                        text=True).strip()
print('FROZEN E7 CODE COMMIT:',SOURCE_COMMIT)
print('Existing previous Kaggle checkout and historical results untouched.')

tests = [
    STUDY/'tests/test_e7_g0_external_source_audit.py',
    STUDY/'tests/test_e7_g0_publish.py',
    STUDY/'tests/test_e7_g0_timestamp_registry.py'
]
subprocess.run([sys.executable,'-m','pytest','-q',*map(str,tests)],check=True)
print('E7 SYNTHETIC TESTS PASSED. Proceed to timestamp inspection.')

## 2. Verify all eight source filenames and preview original timestamp strings

The timestamp parser comes from the **committed per-file registry**, never inferred from test outcomes. No flow rows, IP addresses or endpoint IDs are displayed. The filename-day check protects the day/month/year interpretation; the afternoon clock ambiguity remains flagged.

In [ ]:
import json
import pandas as pd

DATA_ROOT=Path('/kaggle/input/datasets/dsshayan/dscicids')
assert DATA_ROOT.is_dir(), 'Attach the dscicids dataset to Kaggle.'
file_paths=sorted(DATA_ROOT.rglob('*.csv'))
registry=json.loads(REGISTRY.read_text(encoding='utf-8'))
entries=registry['filename_registry']

assert len(file_paths)==registry['required_csv_count']==8
assert len({str(p.parent) for p in file_paths})==1
assert file_paths[0].parent.name==registry['actual_csv_parent_basename']
assert {p.name for p in file_paths}==set(entries), 'Unexpected or missing source file: stop.'
print('EXACT DATASET DIRECTORY:',repr(str(file_paths[0].parent)))
print('DATE REGISTRY:',registry['version'])
CLOCK_WARNINGS=set()

for i,p in enumerate(file_paths,1):
    spec=entries[p.name]
    samples=pd.read_csv(p,nrows=20,dtype='string',encoding='utf-8-sig',
       usecols=lambda header:header.strip().lower()=='timestamp')
    assert samples.shape[1]==1, f'{p.name}: Timestamp column absent/ambiguous'
    originals=samples.iloc[:,0].astype('string').str.strip()
    times=pd.to_datetime(originals,format=spec['timestamp_format'],errors='coerce',exact=True)
    assert times.notna().all(), f'{p.name}: sample has unparseable timestamps'
    observed=set(times.dt.day_name())
    assert observed=={spec['expected_weekday']}, f'{p.name}: sample weekday conflict'
    print(f'[{i}/8] {p.name}')
    print('  Original timestamp strings:',originals.head(3).tolist())
    print('  Frozen parser:',spec['timestamp_format'],'| day:',spec['expected_weekday'])
    if 'Afternoon' in p.name and any(times.dt.hour<12):
        CLOCK_WARNINGS.add(p.name)
        print('  CLOCK SEMANTICS WARNING: Afternoon filename but raw hours below 12; AM/PM absent.')

print('ALL 8 FILES MATCH SOURCE REGISTRY AND SAMPLED CALENDAR WEEKDAYS')
print('Clock warnings:',len(CLOCK_WARNINGS),'— NOT automatically converted to 24-hour time.')

## 3. Authenticate GitHub backup access **before scanning the first large CSV**

Requires existing Kaggle secret \`pushRare\` with repository write permission and Internet enabled. The secret value is never displayed. Uploads only safe per-source G0 summaries; fitted models/flow rows/IP addresses are not published.

In [ ]:
from kaggle_secrets import UserSecretsClient
import importlib.util

def load_research_module(name,path):
    definition=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(definition)
    definition.loader.exec_module(module)
    return module

audit=load_research_module('e7_g0_original_audit',SCRIPT)
publish=load_research_module('e7_g0_public_archive',PUBLISHER)
token=UserSecretsClient().get_secret('pushRare')
assert token, 'Kaggle pushRare secret required before the full scan.'
publish.github_json('GET',
    'https://api.github.com/repos/itsCodeBakery/rare-attack-spatiotemporal-ids',
    token)
print('GITHUB TOKEN AND NETWORK VERIFIED (secret value hidden).')
print('PUBLIC TARGET:',publish.PREFIX)

## 4. Full scan and verified public backup **after every file**

Each file is scanned using its own registered timestamp format. Reports are saved locally **immutably**, then uploaded and independently read back from GitHub. If a file contains parsing or endpoint-quality problems or inconsistent weekday, the public record explicitly becomes \`G0_NOT_CLEARED\`. The clock-ambiguity warnings are also recorded as blockers, not "fixed" by guessing PM. We continue auditing the other files so all eight sources receive original, honest status reports; we do **not** start training.

An interrupted notebook can resume: existing local results are accepted only if original SHA-256, source name, format and full-scan status match. If a GitHub record differs, the upload refuses to overwrite it.

In [ ]:
OUT=Path('/kaggle/working/E7-EXT-VALIDITY-001/G0')
OUT.mkdir(parents=True,exist_ok=True)
all_rows=0
blocked=[]
receipts=[]

for i,p in enumerate(file_paths,1):
    params=entries[p.name]
    report_path=OUT/(p.name+'.audit.json')
    print()
    print(f'G0 [{i}/8] AUDITING {p.name}',flush=True)

    if report_path.exists():
        report=json.loads(report_path.read_text(encoding='utf-8'))
        assert report['external_source_basename']==p.name
        assert report['input_artifact_sha256']==audit.fingerprint(p)
        assert report['timestamp_format_declared']==params['timestamp_format']
        assert report.get('full_file_scan') is True
        print('Original local full-scan checkpoint verified.',flush=True)
    else:
        report=audit.inspect(p,timestamp_format=params['timestamp_format'],
                             max_rows=0,chunk_size=40000)
        item=report['files'][0]
        observed_dates={}
        for date_label,n in item['label_day_counts'].items():
            day=date_label.split(' | ',1)[0]
            observed_dates[day]=observed_dates.get(day,0)+int(n)
        invalid_days=[day for day in observed_dates
                      if day=='__BAD_TIME__' or
                         pd.Timestamp(day).day_name()!=params['expected_weekday']]

        if invalid_days:
            report['explicit_followup_gates'].append(
                'Source file weekday conflict: expected '+params['expected_weekday']+
                '; review source day/date field before any model training')
        if p.name in CLOCK_WARNINGS:
            report['explicit_followup_gates'].append(
                'CLOCK_AMPM_AMBIGUITY: Filename says Afternoon, preview contains bare '
                '1-11 hour without AM/PM; event clock time not authenticated. '
                'Do not silently add 12 hours without independent source evidence.')

        if invalid_days or p.name in CLOCK_WARNINGS:
            item['status']='DATA_QUALITY_REVIEW_REQUIRED'
            report['status']='G0_NOT_CLEARED'

        report_text=json.dumps(report,indent=2,sort_keys=True,ensure_ascii=False)+'\n'
        temporary=report_path.with_suffix(report_path.suffix+'.tmp')
        temporary.write_text(report_text,encoding='utf-8')
        temporary.replace(report_path)
        print('Original source audit checkpoint saved locally.',flush=True)

    # Public G0 schema prohibits raw network records/IP values; exact remote bytes checked.
    uploaded=publish.publish_report(report_path,token=token,run_path=publish.PREFIX)
    assert uploaded['status'] in ('UPLOADED_AND_VERIFIED','ALREADY_SAVED_IDENTICALLY')
    receipts.append(uploaded)
    print('GITHUB BACKUP VERIFIED:',uploaded['status'],flush=True)
    print('PUBLIC FILE: https://github.com/'+publish.REPO+
          '/blob/main/'+uploaded['remote'],flush=True)

    item=report['files'][0]
    all_rows+=int(item['rows_inspected'])
    print('Rows:',item['rows_inspected'],
          '| Classes:',len(item['labels']),
          '| Unparsed timestamps:',item['parse_failures'],
          '| Timestamp row-order reversals:',item['timestamp_disorder_adjacent'])
    print('SOURCE AUDIT STATUS:',report['status'],flush=True)
    if report['status']!='G0_SCHEMA_CHECKED_TEMPORAL_VALIDITY_PENDING':
        blocked.append(p.name)

print()
print('='*75)
print('E7 G0: ORIGINAL SOURCE REPORTS VERIFIED ON GITHUB:',len(receipts),'/8')
print('Total observed source rows:',all_rows)
print('Number of files requiring clock/data-quality review:',len(blocked))
print('Review-required filenames:',blocked)
print('PUBLIC SOURCE EVIDENCE: https://github.com/'+publish.REPO+
      '/tree/main/'+publish.PREFIX)
print('NO MODEL FITTING. NO ORIGINAL UNSW TEST ACCESS. NO VERIFIED CAUSAL ORDER.')
print('DO NOT ADVANCE TO MODEL TRAINING.')
print('Send this complete summary to ChatGPT for Notebook 27.')
print('='*75)

## Stop gate / what follows

Even if every CSV parses, calendar dates and row ordering are **not** authenticated flow chronology. CIC-IDS2017 \`TrafficLabelling\` archives have known labeling/flow-generation issues; timestamps may have 12h ambiguity despite a valid 24h parser. **No G0 report alone clears causal time use.**

The next notebook, **27**, will investigate ambiguous clock hours, flow duration/start/end units, source-target endpoint linkage, duplicates and per-day attack support, while continuing the same GitHub-audited workflow. Only then can training/held-out evaluation be preregistered.

Source and earlier experiment evidence remain unchanged. This is a new E7 G0-only run.